# Análises do Capítulo 5 — lote 1 (Análise 1: característica do problema × característica do algoritmo)

**SPEC**: `SPEC_implementacao_analises_cap5.md` v0.3 (§4) · **catálogo**: fichas 17, 18, 19, 20, 21, 22, 23, 24, 25, 54, 75 · **matriz de características**: `caracteristicas_v3.csv` (ratificada em 07/09/2026, SPEC §4.2-A).

**Estado deste caderno: PROTÓTIPO.** As células de análise rodam sobre o cache de 17/08 (`data/analysis_cache/metricas_endpoint_snapshot_2026-08-17.parquet`), filtrado pelo censo canônico de 22/08 (`f5/final/censo_final.csv`). Os números **não são publicáveis** até a célula de consolidação (§1.0-C, a rodar no Mac sobre o corpus bruto) gerar `cache_endpoint.parquet` — em particular: o IBEA-MS (e103) não tem ⑦ no cache de 17/08 (0 células válidas aqui); o DDMOP7 tem só o qNEHVI com 30 sementes no cache; a régua do DDMOP7 no cache é a provisória.

**Regras obedecidas (SPEC §1)**: C6 (censo 22/08), C8 parcial (só células a orçamento pleno — `status == ok`), A9 (e81 × ESTOQUE40 sementes 22–26 fora), DDMOP7 só HV, e103 fora do DDMOP7, D25 (casamento assistido × piso), D70 (IGD+ primária, HV secundária), duas réguas em toda seção (ranking absoluto + Δ pareado vs piso casado). Nomes de exibição em toda figura/tabela (nunca IDs internos).

**Duas réguas**: ranking absoluto = rank por problema entre as 17 configurações *online* (13 assistidos + 4 pisos), *offline* à parte; ganho sobre o piso = Δ pareado por semente, Δ = (IGD+_piso − IGD+_assistido)/IGD+_piso (> 0: o assistido é melhor), com a taxa de vitórias por semente; para JES, qPOTS e LBN-MOBO o piso é o melhor dos quatro pisos na mesma semente (banda, ficha 10). HV: Δ_HV = (HV_assistido − HV_piso)/HV_piso (régua secundária; a única no DDMOP7).

Itens marcados **[proposta]** são do agente e não valem até o autor decidir (SPEC §5).

In [1]:
# ── 0. Setup ────────────────────────────────────────────────────────────────
import os, json, warnings, itertools
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import scipy.stats as st
warnings.filterwarnings("ignore", category=FutureWarning)

# Caminhos (no Mac: UA = raiz do repositório ua-dd-saea; CARAC = onde a matriz for guardada)
UA    = os.environ.get("UA_ROOT", "/mnt/user-data/uploads/ua-dd-saea")
CACHE = os.path.join(UA, "data", "analysis_cache")
CENSO = os.path.join(UA, "f5", "final", "censo_final.csv")
CARAC = os.environ.get("CARAC_CSV", "/home/claude/mineracao/out/caracteristicas_v3.csv")
OUT   = os.environ.get("OUT_DIR", "/home/claude/notebook/out")
FIGDIR, TABDIR = os.path.join(OUT, "figures"), os.path.join(OUT, "tables")
os.makedirs(FIGDIR, exist_ok=True); os.makedirs(TABDIR, exist_ok=True)

# Configurações (IDs internos só aqui; toda saída usa NOME)
NOME = {"b1": "ParEGO", "b3": "K-RVEA", "b4": "CSEA", "c122": "θ-DEA-DP",
        "c141": "MMRAEA", "c149": "LBN-MOBO", "c154": "JES", "c217": "PC-SAEA",
        "c238": "EIM", "c262": "qNEHVI", "e7": "EDN-ARMOEA", "e74": "CLMEA",
        "e81": "qPOTS", "nsga2": "NSGA-II", "nsga3": "NSGA-III", "moead": "MOEA/D",
        "smsemoa": "SMS-EMOA", "b5m": "Prob-MOEA/D", "b5r": "Prob-RVEA",
        "e103": "IBEA-MS", "moead_media": "MOEA/D-média"}
SA_ON   = ["b4", "c217", "e74", "c141", "c238", "b3", "c122", "b1", "e7", "c262", "c154", "e81", "c149"]
PISOS   = ["nsga2", "moead", "nsga3", "smsemoa"]
ONLINE  = SA_ON + PISOS                                # 17 configurações online
PISO_DE = {"b4": "nsga2", "c217": "nsga2", "e74": "nsga2", "c141": "nsga2", "c238": "nsga2",
           "b3": "moead", "c122": "moead", "b1": "moead", "e7": "nsga3", "c262": "smsemoa"}   # D25
BANDA   = ["c154", "e81", "c149"]                      # contra o melhor dos 4 pisos na semente
SA_OFF, PISO_OFF = ["b5m", "b5r", "e103"], "moead_media"
OFFLINE = SA_OFF + [PISO_OFF]

# Eixos da taxonomia (tab:roster do cap. 4) e famílias da §3.8 (tab:familias-depara; nomes F1–F7 provisórios)
FUNCAO  = {"b1": "OV", "c238": "OV", "c262": "OV", "c149": "OV", "e81": "OV", "c141": "AV",
           "b3": "OM", "e7": "OM", "e74": "OM", "b4": "AM", "c217": "AM", "e103": "AM",
           "c154": "OL", "c122": "AR", "b5r": "AR", "b5m": "AR"}
MOTOR   = {"b1": "BO·Dec", "c238": "BO·MP", "b3": "EA·Dec", "b4": "EA·Dom", "c262": "BO·HV",
           "c122": "EA·Dec", "b5r": "EA·Dec", "b5m": "EA·Dec", "e7": "EA·Ind", "c154": "BO·Esp",
           "c217": "EA·Dom", "e74": "EA·Dom", "e103": "EA·Ind", "c149": "BO·Esp", "c141": "EA·Dom",
           "e81": "BO·Esp"}
SURR    = {"b1": "GP", "c238": "GP", "b3": "GP", "b4": "CL", "c262": "GP", "c122": "CL", "b5r": "GP",
           "b5m": "GP", "e7": "NN", "c154": "GP", "c217": "CL", "e74": "RG", "e103": "GP",
           "c149": "NN", "c141": "RG", "e81": "GP"}
MEDICAO = {"b1": "VA", "c238": "VA", "b3": "VA", "b4": "EE", "c262": "VA", "c122": "VN", "b5r": "VA",
           "b5m": "VA", "e7": "DE", "c154": "VA", "c217": "EE", "e74": "DG", "e103": "VA",
           "c149": "DE", "c141": "DE", "e81": "VA"}
FAMILIA = {"b1": "F1", "c238": "F1", "c154": "F1", "c262": "F1", "e81": "F1", "b3": "F2", "e103": "F2",
           "c141": "F3", "c149": "F3", "e7": "F3", "b4": "F4", "c217": "F4", "c122": "F5",
           "b5r": "F6", "b5m": "F6", "e74": "F7"}
FUNCAO_NOME = {"OV": "OV · aquisição exploratória", "AV": "AV · penalização de risco",
               "OM": "OM · aprendizado ativo", "AM": "AM · gestão de confiança",
               "OL": "OL · ganho de informação", "AR": "AR · dominância probabilística"}
# Paleta categórica (6 classes da função da incerteza; validada com o validador do skill dataviz, modo claro)
COR_FUNCAO = {"OV": "#2a78d6", "AV": "#eb6834", "OM": "#1baf7a", "AM": "#eda100", "OL": "#e87ba4", "AR": "#008300"}
VERDE  = LinearSegmentedColormap.from_list("verde", ["#fbfdfb", "#7bcf92"])
DIVERG = LinearSegmentedColormap.from_list("div", ["#dfa07e", "#fbfbf9", "#7bcf92"])
CINZA_PISO = "#8f8f8f"

PROBLEMAS = ["ZDT1", "ZDT3", "ZDT4", "ZDT6", "DTLZ1", "DTLZ2", "DTLZ3", "DTLZ4", "DTLZ7",
             "WFG1", "WFG2", "WFG4", "WFG5", "WFG9", "MMF1", "MMF4", "MMF11_L", "MMF16_20",
             "BBOB_F1", "BBOB_F5", "BBOB_F17", "BBOB_F22", "BBOB_F37", "BBOB_F49", "BBOB_F55",
             "RE21", "ESTOQUE40", "DDMOP7"]
SINTETICOS, REAIS = PROBLEMAS[:25], PROBLEMAS[25:]
SEM_IGDP = {"DDMOP7"}                                  # D72/C12: só HV
ROTULO_PROB = {p: p.replace("BBOB_", "") for p in PROBLEMAS}   # rótulo curto nas figuras

def salva_fig(fig, nome):
    for ext in ("pdf", "png"):
        fig.savefig(os.path.join(FIGDIR, f"{nome}.{ext}"), dpi=220, bbox_inches="tight")
    plt.close(fig)

def salva_tab(df, nome, **kw):
    df.to_csv(os.path.join(TABDIR, f"{nome}.csv"), **kw)

def limpa(ax):
    ax.tick_params(length=0)
    for sp in ax.spines.values():
        sp.set_visible(False)

YLIM_DELTA = (-100, 100)     # Δ relativo é ilimitado por baixo; a figura corta em −100 % e marca o corte

def rotula_fim(ax, itens, fs=4.8, gap_frac=0.035):
    # Rótulos no fim da linha, empurrados para não se sobrepor (itens = [(x, y, texto, cor)])
    if not itens: return
    lo, hi = ax.get_ylim(); gap = (hi - lo) * gap_frac
    itens = sorted(itens, key=lambda t: t[1]); ys = [max(lo, min(hi, t[1])) for t in itens]
    for i in range(1, len(ys)):
        if ys[i] - ys[i - 1] < gap: ys[i] = ys[i - 1] + gap
    exc = ys[-1] - hi
    if exc > 0: ys = [y - exc for y in ys]
    for (x, y, txt, cor), yy in zip(itens, ys):
        ax.annotate(txt, (x, yy), xytext=(3, 0), textcoords="offset points", fontsize=fs, va="center", color=cor)

def linhas_escada(ax, Dm, x, algs, rotulos=True):
    # Linhas finas por algoritmo (cor = classe) + grossa por classe; corte em YLIM_DELTA com marcador
    ax.axhline(0, color="#1a1a1a", lw=.8); fins = []
    for a in algs:
        y = Dm[a].values.astype(float) * 100
        if np.isnan(y).all(): continue
        yc = np.clip(y, *YLIM_DELTA)
        ax.plot(x, yc, color=COR_FUNCAO[FUNCAO[a]], lw=.9, alpha=.55)
        cortados = np.where(y < YLIM_DELTA[0])[0]
        if len(cortados): ax.plot(x[cortados], yc[cortados], "v", color=COR_FUNCAO[FUNCAO[a]], ms=3, alpha=.7)
        k = np.where(~np.isnan(y))[0][-1]; fins.append((x[k], yc[k], NOME[a], "#333333"))
    for cl in ["OV", "AV", "OM", "AM", "OL", "AR"]:
        aa = [a for a in algs if FUNCAO[a] == cl]
        y = Dm[aa].median(axis=1).values.astype(float) * 100
        if not np.isnan(y).all(): ax.plot(x, np.clip(y, *YLIM_DELTA), color=COR_FUNCAO[cl], lw=2.2, label=FUNCAO_NOME[cl])
    ax.set_ylim(*YLIM_DELTA)
    if rotulos: rotula_fim(ax, fins)

print("setup ok ·", len(ONLINE), "online ·", len(OFFLINE), "offline ·", len(PROBLEMAS), "problemas")

setup ok · 17 online · 4 offline · 28 problemas


## 1.0-C · Consolidação (a rodar no Mac sobre o corpus bruto — **não roda aqui**)

Lê a camada ① de cada célula válida do censo (e a ⑦ no *offline*), mede IGD+/HV/n_nd com `src.metrics.metrics_of_set` (régua S.5, ref 1,1, `checa_regua`) e grava `data/analysis_cache/cache_endpoint.parquet` com as colunas `exp, alg, problema, semente, igd_plus, hv, igd, gd, spacing, n_nd, n_aval, status, motivo`. O gate `hv_smoke_bbob_f1() == 1,0433` é conferido antes de qualquer medida. Enquanto esse arquivo não existe, a célula seguinte cai no cache de 17/08 (protótipo).

In [2]:
# ── 1.0-C. Consolidação ① → cache_endpoint (só executa se RODAR_CONSOLIDACAO=1) ────────────
# Uma passada pelo corpus bruto, no Mac. Para cada célula ok do censo (exp main/off):
#   lê a ① (__real.parquet; no offline a ⑦ __final.parquet — §D9), separa o conjunto não dominado,
#   mede IGD+/HV/IGD/GD/spacing/n_nd com src.metrics (régua S.5, ref 1,1, checa_regua) e guarda 1 linha;
#   exporta o ND (x e f) por problema — insumo das fotos (1D), do IGDX nos MMF (1C-25) e dos fenótipos (1F);
#   exporta as frentes verdadeiras (F) e, nos MMF, os conjuntos de Pareto verdadeiros (X) de problems.py.
RODAR_CONSOLIDACAO = os.environ.get("RODAR_CONSOLIDACAO", "0") == "1"
CACHE_END = os.path.join(CACHE, "cache_endpoint.parquet")
NDDIR = os.path.join(CACHE, "nd")

def nd_mask(F):
    F = np.asarray(F, float); n = F.shape[0]; keep = np.ones(n, bool)
    for i in range(n):
        if not keep[i]: continue
        dom = np.all(F <= F[i], axis=1) & np.any(F < F[i], axis=1)
        if dom.any(): keep[i] = False
    return keep

if RODAR_CONSOLIDACAO:
    import sys, time; sys.path.insert(0, UA)
    import pyarrow.parquet as pq
    from src import metrics, naming, experiment
    DATA_ROOT = os.environ.get("DATA_ROOT", naming.DEFAULT_DATA_ROOT)
    def caminho_camada(exp, alg, problema, semente, camada):
        # dois leiautes do corpus: o do repo (data/experiments/{exp}/{alg}/…, via naming.layer_path)
        # e o da pasta resultados_experimentos ({alg}/{problema}/{semente}/…); o nome do arquivo é o mesmo
        cands = [naming.layer_path(exp, alg, problema, semente, camada, DATA_ROOT),
                 os.path.join(DATA_ROOT, alg, problema, str(semente), f"exp_{exp}_{alg}_{problema}_{semente}__{camada}.parquet"),
                 naming.layer_path(exp, alg, problema, semente, camada, os.path.join(UA, naming.DEFAULT_DATA_ROOT))]   # o repo guarda a semente 42 no seu próprio leiaute
        for c in cands:
            if os.path.exists(c): return c
        raise FileNotFoundError(" | ".join(cands))
    os.makedirs(NDDIR, exist_ok=True)
    gate = metrics.hv_smoke_bbob_f1()
    assert abs(gate - metrics.HV_SMOKE_BBOB_F1) < 5e-4, f"gate D92 falhou: {gate}"
    print(f"gate hv_smoke_bbob_f1 = {gate:.4f} ok")
    censo = pd.read_csv(CENSO)
    censo = censo[censo.exp.isin(["main", "off"])]        # sweep-*/batch ficam fora (fichas 63/64 descartadas)
    linhas, refs, nd_por_prob = [], {}, {}
    t0 = time.time()
    for k, r in enumerate(censo.itertuples(index=False)):
        rec = dict(exp=r.exp, alg=r.alg, problema=r.problema, semente=int(r.semente),
                   status=r.status, motivo=r.motivo, igd_plus=np.nan, hv=np.nan, igd=np.nan,
                   gd=np.nan, spacing=np.nan, n_nd=0, n_aval=0)
        if r.status == "ok":
            try:
                camada = "final" if r.exp == "off" else "real"                      # §D9: endpoint offline = ⑦
                tbl = pq.read_table(caminho_camada(r.exp, r.alg, r.problema, r.semente, camada))
                fcols = metrics._f_columns(tbl.column_names)
                xcols = [c for c in tbl.column_names if c.startswith("x") and c[1:].isdigit()]
                F = np.column_stack([np.asarray(tbl.column(c), np.float64) for c in fcols])
                X = np.column_stack([np.asarray(tbl.column(c), np.float64) for c in xcols]) if xcols else None
                rec["n_aval"] = int(F.shape[0])
                m = nd_mask(F); rec["n_nd"] = int(m.sum())
                if r.problema in metrics.PROBLEMAS_SEM_FRONT_D72:                  # DDMOP7: só HV
                    ideal, nadir = metrics.reference_bounds(r.problema); metrics.checa_regua(F, r.problema)
                    rec["hv"] = metrics.hv(metrics.normalize(F[m], ideal, nadir))
                else:
                    if r.problema not in refs: refs[r.problema] = metrics.reference_set(r.problema)
                    rec.update(metrics.metrics_of_set(F, r.problema, ref_norm=refs[r.problema]))
                nd = pd.DataFrame(F[m], columns=fcols); nd.insert(0, "semente", int(r.semente))
                nd.insert(0, "alg", r.alg); nd.insert(0, "exp", r.exp)
                if X is not None:
                    for j, c in enumerate(xcols): nd[c] = X[m, j]
                nd_por_prob.setdefault(r.problema, []).append(nd)
            except Exception as e:                    # pára-e-loga por célula (D81): registra, não inventa
                rec["motivo"] = f"erro_metrica:{type(e).__name__}:{str(e)[:80]}"
        linhas.append(rec)
        if (k + 1) % 1000 == 0: print(f"  {k+1}/{len(censo)} células · {time.time()-t0:.0f} s")
    pd.DataFrame(linhas).to_parquet(CACHE_END, index=False)
    for p, lst in nd_por_prob.items():
        pd.concat(lst, ignore_index=True).to_parquet(os.path.join(NDDIR, f"nd_{p}.parquet"), index=False)
    # frentes verdadeiras (F) e conjuntos de Pareto verdadeiros dos MMF (X) — só existem no Mac (problems.py)
    fr, fr_erros = [], []
    for p in PROBLEMAS:
        if p in metrics.PROBLEMAS_SEM_FRONT_D72: continue
        try:
            prob = experiment._instantiate_problem(p); Xv, Fv = prob.true_pareto_front(5000)
            d = pd.DataFrame(np.asarray(Fv, float), columns=[f"f{i}" for i in range(np.asarray(Fv).shape[1])]); d.insert(0, "problema", p); fr.append(d)
            if p.startswith("MMF") and Xv is not None:
                pd.DataFrame(np.asarray(Xv, float), columns=[f"x{i}" for i in range(np.asarray(Xv).shape[1])]).to_parquet(os.path.join(NDDIR, f"ps_verdadeiro_{p}.parquet"), index=False)
        except Exception as e:                        # não derruba a consolidação: registra e segue
            fr_erros.append(f"{p}: {type(e).__name__}: {str(e)[:80]}")
    if fr: pd.concat(fr, ignore_index=True).to_parquet(os.path.join(CACHE, "frentes_verdadeiras.parquet"), index=False)
    if fr_erros: print("frentes verdadeiras NÃO exportadas para:", fr_erros)
    erros = sum(1 for l in linhas if str(l["motivo"]).startswith("erro_metrica"))
    print(f"cache_endpoint gravado: {len(linhas)} linhas · {erros} células com erro de métrica (ver coluna motivo) · "
          f"ND exportado para {len(nd_por_prob)} problemas · {time.time()-t0:.0f} s")
else:
    print("consolidação não executada (RODAR_CONSOLIDACAO≠1); usa o cache disponível")

gate hv_smoke_bbob_f1 = 1.0433 ok


  1000/16667 células · 524 s


  2000/16667 células · 536 s


  3000/16667 células · 551 s


  4000/16667 células · 568 s


...

  13000/16667 células · 665 s


  14000/16667 células · 679 s


  15000/16667 células · 695 s


  16000/16667 células · 708 s


cache_endpoint gravado: 16667 linhas · 120 células com erro de métrica (ver coluna motivo) · ND exportado para 28 problemas · 1217 s


## 1.0 · Endpoint canônico e Δ pareado (base de todas as análises)

In [3]:
# ── 1.0. Carregar endpoint + filtros do censo + verificações embutidas ───────────────────
censo = pd.read_csv(CENSO)
censo = censo[censo.exp.isin(["main", "off"])]            # sweep-*/batch ficam fora (fichas 63/64 descartadas); sementes 0–28 e 42
if os.path.exists(CACHE_END):
    END = pd.read_parquet(CACHE_END); FONTE_END = "cache_endpoint.parquet (consolidação canônica)"
else:
    END = pd.read_parquet(os.path.join(CACHE, "metricas_endpoint_snapshot_2026-08-17.parquet"))
    END = END.merge(censo[["alg", "exp", "problema", "semente", "status", "motivo"]],
                    on=["alg", "exp", "problema", "semente"], how="left")
    END = END[END.erro.fillna("") == ""]              # células sem métrica no cache ficam fora
    FONTE_END = "cache de 17/08 filtrado pelo censo de 22/08 (PROTÓTIPO)"
END = END[END.status == "ok"].copy()                    # C6 + C8 parcial (orçamento pleno)
END = END[~((END.alg == "e81") & (END.problema == "ESTOQUE40") & END.semente.between(22, 26))]   # A9
END = END[~((END.alg == "e103") & (END.problema == "DDMOP7"))]                                  # B1(a)
END.loc[END.problema.isin(SEM_IGDP), "igd_plus"] = np.nan                                       # D72: só HV
END = END[END.alg.isin(ONLINE + OFFLINE)]
print("fonte:", FONTE_END)
print(f"células válidas: {len(END)} (censo ok = {(censo.status=='ok').sum()}; "
      f"fora do plantel/pisos no censo = {(~censo.alg.isin(ONLINE+OFFLINE)).sum()})")

# verificação: n de células por configuração vs censo
chk = (END.groupby("alg").size().rename("no_cache").reindex(ONLINE + OFFLINE).fillna(0).astype(int)
       .to_frame().join(censo[censo.status == "ok"].groupby("alg").size().rename("censo_ok")))
chk["faltam"] = chk.censo_ok - chk.no_cache
chk.index = [NOME[a] for a in chk.index]
print(chk.sort_values("faltam", ascending=False).to_string())
if chk.faltam.sum() > 0:
    print(f"\n⚠ {int(chk.faltam.sum())} células ok no censo sem métrica nesta fonte — protótipo; "
          "a consolidação (1.0-C) fecha a diferença.")
# métricas ausentes (NaN) por regime e motivos de erro da consolidação, se houver
sem = END.groupby("exp").agg(n=("alg", "size"), com_igdp=("igd_plus", lambda x: int(x.notna().sum())), com_hv=("hv", lambda x: int(x.notna().sum())))
print("\nmétricas presentes por regime:"); print(sem.to_string())
if "motivo" in END.columns:
    err = END[END.motivo.astype(str).str.startswith("erro_metrica")]
    if len(err):
        print(f"\n⚠ {len(err)} células com erro de métrica na consolidação (não inventadas):")
        print(err.motivo.astype(str).str[:110].value_counts().head(8).to_string())
        print(err.groupby(["exp", "alg"]).size().rename("n").to_string())

fonte: cache_endpoint.parquet (consolidação canônica)
células válidas: 15828 (censo ok = 15883; fora do plantel/pisos no censo = 25)
              no_cache  censo_ok  faltam
IBEA-MS            728       758      30
CSEA               791       791       0
qPOTS              784       784       0
Prob-RVEA          815       815       0
Prob-MOEA/D        709       709       0
SMS-EMOA           840       840       0
NSGA-III           840       840       0
MOEA/D             840       840       0
NSGA-II            835       835       0
LBN-MOBO           735       735       0
JES                259       259       0
PC-SAEA            840       840       0
qNEHVI             624       624       0
EDN-ARMOEA         724       724       0
ParEGO             732       732       0
θ-DEA-DP           700       700       0
K-RVEA             804       804       0
EIM                734       734       0
MMRAEA             840       840       0
CLMEA              839       839       0
MOEA/D

In [4]:
# ── 1.0. Medianas por célula, rank por problema, Δ pareado por semente ───────────────────
on, off = END[END.exp == "main"], END[END.exp == "off"]

def medianas(base, metrica, algs):
    g = base[base.alg.isin(algs)].groupby(["problema", "alg"])[metrica]
    med = g.median().unstack().reindex(index=PROBLEMAS, columns=algs)
    n = g.count().unstack().reindex(index=PROBLEMAS, columns=algs).fillna(0).astype(int)
    med = med.where(n >= 5)                              # ≥ 5 sementes para publicar uma mediana
    return med, n

MED_IGDP, N_IGDP = medianas(on, "igd_plus", ONLINE)
MED_HV,   N_HV   = medianas(on, "hv", ONLINE)
MED_IGDP_OFF, N_OFF = medianas(off, "igd_plus", OFFLINE)
MED_HV_OFF, _ = medianas(off, "hv", OFFLINE)
RANK     = MED_IGDP.rank(axis=1, method="average")      # régua 1: rank por problema entre os 17 online (IGD+)
RANK_HV  = (-MED_HV).rank(axis=1, method="average")
RANK_OFF = MED_IGDP_OFF.rank(axis=1, method="average")

def delta_pareado(base, sa_list, piso_de, banda, pisos, metrica, maior_melhor):
    linhas = []
    for a in sa_list:
        sa = base[base.alg == a][["problema", "semente", metrica]].dropna()
        for p, sp in sa.groupby("problema"):
            if a in piso_de:
                pi = base[(base.alg == piso_de[a]) & (base.problema == p)][["semente", metrica]].dropna()
                pi = pi.rename(columns={metrica: "piso"})
            else:                                        # banda: melhor dos 4 pisos na mesma semente
                pp = base[base.alg.isin(pisos) & (base.problema == p)][["semente", metrica]].dropna()
                agg = "max" if maior_melhor else "min"
                pi = pp.groupby("semente")[metrica].agg(agg).rename("piso").reset_index()
            j = sp.merge(pi, on="semente")
            if not len(j):
                continue
            if maior_melhor:
                d = (j[metrica] - j.piso) / j.piso.where(j.piso.abs() > 1e-12)
            else:
                d = (j.piso - j[metrica]) / j.piso.where(j.piso.abs() > 1e-12)
            for s, dv in zip(j.semente, d):
                linhas.append(dict(alg=a, problema=p, semente=int(s), delta=float(dv),
                                   vitoria=bool(dv > 0), piso=piso_de.get(a, "banda")))
    return pd.DataFrame(linhas, columns=["alg", "problema", "semente", "delta", "vitoria", "piso"])

D_IGDP = delta_pareado(on, SA_ON, PISO_DE, BANDA, PISOS, "igd_plus", False)
D_HV   = delta_pareado(on, SA_ON, PISO_DE, BANDA, PISOS, "hv", True)
D_OFF  = delta_pareado(off, SA_OFF, {a: PISO_OFF for a in SA_OFF}, [], [], "igd_plus", False)
D_OFF_HV = delta_pareado(off, SA_OFF, {a: PISO_OFF for a in SA_OFF}, [], [], "hv", True)

TESTE_PAREADO_PROPOSTA = True   # [proposta] Wilcoxon pareado por semente por célula; coluna informativa, não usada nas figuras
def por_celula(D):
    if D.empty:
        return pd.DataFrame(columns=["alg", "problema", "delta_med", "taxa_vit", "n_pares", "p_wilcoxon_proposta"])
    g = D.groupby(["alg", "problema"])
    out = g.agg(delta_med=("delta", "median"), taxa_vit=("vitoria", "mean"), n_pares=("delta", "size")).reset_index()
    if TESTE_PAREADO_PROPOSTA:
        pv = {}
        for (a, p), s in g:
            x = s.delta.values
            pv[(a, p)] = st.wilcoxon(x).pvalue if (len(x) >= 5 and np.any(x != 0)) else np.nan
        out["p_wilcoxon_proposta"] = [pv[(a, p)] for a, p in zip(out.alg, out.problema)]
    out = out[out.n_pares >= 5]
    return out
DCEL, DCEL_HV, DCEL_OFF, DCEL_OFF_HV = por_celula(D_IGDP), por_celula(D_HV), por_celula(D_OFF), por_celula(D_OFF_HV)

# saídas da 1.0
for nome, df in [("tab_1_0_mediana_igdp_online", MED_IGDP), ("tab_1_0_n_sementes_online", N_IGDP),
                 ("tab_1_0_mediana_hv_online", MED_HV), ("tab_1_0_mediana_igdp_offline", MED_IGDP_OFF),
                 ("tab_1_0_rank_igdp_online", RANK)]:
    salva_tab(df.rename(columns=NOME), nome)
for nome, df in [("cache_delta_igdp_online", D_IGDP), ("cache_delta_hv_online", D_HV),
                 ("cache_delta_igdp_offline", D_OFF), ("tab_1_0_delta_celula_igdp", DCEL),
                 ("tab_1_0_delta_celula_hv", DCEL_HV), ("tab_1_0_delta_celula_igdp_offline", DCEL_OFF)]:
    salva_tab(df.assign(alg=df.alg.map(NOME)), nome, index=False)

placar = pd.DataFrame([dict(alg=NOME[a], vence=int((s.delta_med > 0).sum()), perde=int((s.delta_med < 0).sum()),
                            n_prob=len(s), delta_mediano=float(s.delta_med.median())) for a, s in DCEL.groupby("alg")]).set_index("alg") \
         if len(DCEL) else pd.DataFrame(columns=["vence", "perde", "n_prob", "delta_mediano"])
print("placar assistido × piso casado (IGD+; problemas com ≥ 5 pares; Δ>0 = assistido melhor):")
print(placar.sort_values("delta_mediano", ascending=False).round(3).to_string())
print("\nrank médio de IGD+ (17 online, problemas com IGD+):")
print(RANK.mean().sort_values().rename(index=NOME).round(2).to_string())

placar assistido × piso casado (IGD+; problemas com ≥ 5 pares; Δ>0 = assistido melhor):
            vence  perde  n_prob  delta_mediano
alg                                            
θ-DEA-DP       24      2      26          0.639
K-RVEA         22      4      26          0.569
MMRAEA         22      5      27          0.557
ParEGO         19      6      25          0.482
qNEHVI         16      6      22          0.477
EIM            18      9      27          0.283
CLMEA          20      7      27          0.238
JES             7      5      12          0.178
EDN-ARMOEA     14     12      26          0.065
CSEA            7     19      26         -0.120
PC-SAEA         6     21      27         -0.179
qPOTS           8     18      26         -0.557
LBN-MOBO        4     22      26         -1.276

rank médio de IGD+ (17 online, problemas com IGD+):
alg
qNEHVI         3.55
MMRAEA         4.33
θ-DEA-DP       4.92
K-RVEA         6.19
CLMEA          6.59
EIM            7.11
ParEGO         

## 1A · ficha 17 — rank médio e Δ por grupo de característica (duas réguas)

Linhas = os sete grupos da matriz (+ duas linhas de referência: os 25 sintéticos e os 3 reais). Rank: média, sobre os problemas do grupo, do rank por problema entre as 17 configurações *online*; Δ: mediana, sobre os problemas do grupo, do Δ mediano por célula. Cor normalizada por linha; n de problemas impresso. Caveat obrigatório: um problema pode pertencer a mais de um grupo — as linhas não são independentes (SPEC §4.2-A).

In [5]:
# ── 1A. Grupos da matriz ratificada ───────────────────────────────────────────────────
CAR = pd.read_csv(CARAC)
CAR["prob"] = np.where(CAR.suite == "BBOB", "BBOB_" + CAR.problema, CAR.problema)
GRUPOS = {}      # grupo → lista de problemas (pertence ⇔ tem degrau)
NOME_GRUPO = {}
for g, s in CAR[CAR.grupo.isin(list("1234567"))].groupby("grupo"):
    GRUPOS[g] = sorted(set(s.prob), key=PROBLEMAS.index); NOME_GRUPO[g] = s.grupo_nome.iloc[0]
LINHAS_1A = [(f"({g}) {NOME_GRUPO[g]}", GRUPOS[g]) for g in sorted(GRUPOS)] + \
            [("sintéticos (25)", SINTETICOS), ("reais (3)", REAIS)]

def agrega_grupo(rank, dcel, linhas, algs_rank, algs_delta):
    R = pd.DataFrame(index=[l for l, _ in linhas], columns=algs_rank, dtype=float)
    Dm = pd.DataFrame(index=R.index, columns=algs_delta, dtype=float)
    Tv = Dm.copy(); Np = pd.Series(0, index=R.index, dtype=int); Nd = Dm.copy()
    for lab, probs in linhas:
        pr = [p for p in probs if p in rank.index]
        Np[lab] = len([p for p in pr if rank.loc[p].notna().any()])
        R.loc[lab] = rank.loc[pr].mean()
        sub = dcel[dcel.problema.isin(pr)]
        for a in algs_delta:
            s = sub[sub.alg == a]
            if len(s):
                Dm.loc[lab, a] = s.delta_med.median(); Tv.loc[lab, a] = s.taxa_vit.mean(); Nd.loc[lab, a] = len(s)
    return R, Dm, Tv, Np, Nd

R1A, D1A, T1A, NP1A, ND1A = agrega_grupo(RANK, DCEL, LINHAS_1A, ONLINE, SA_ON)
R1A_HV, D1A_HV, T1A_HV, _, _ = agrega_grupo(RANK_HV, DCEL_HV, LINHAS_1A, ONLINE, SA_ON)
salva_tab(R1A.rename(columns=NOME).assign(n_problemas=NP1A), "tab_1A_rank_grupo")
salva_tab(D1A.rename(columns=NOME), "tab_1A_delta_grupo")
salva_tab(T1A.rename(columns=NOME), "tab_1A_taxa_vitorias_grupo")
salva_tab(R1A_HV.rename(columns=NOME).assign(n_problemas=NP1A), "tab_1A_rank_grupo_hv")
salva_tab(D1A_HV.rename(columns=NOME), "tab_1A_delta_grupo_hv")

def heat_linhas(ax, M, fmt, cmap, invert, colunas_nome, sep_apos=None, fs=6.2, texto=None):
    V = M.values.astype(float); C = np.full(V.shape, np.nan)
    for i in range(V.shape[0]):
        r = V[i]; okm = ~np.isnan(r)
        if okm.sum() > 1 and np.nanmax(r) > np.nanmin(r):
            C[i, okm] = (r[okm] - np.nanmin(r)) / (np.nanmax(r) - np.nanmin(r))
        elif okm.any():
            C[i, okm] = .5
    if invert: C = 1 - C
    ax.imshow(np.ma.masked_invalid(C), cmap=cmap, vmin=0, vmax=1, aspect="auto")
    for i in range(V.shape[0]):
        for j in range(V.shape[1]):
            if np.isnan(V[i, j]):
                ax.add_patch(plt.Rectangle((j - .5, i - .5), 1, 1, color="#ececec", hatch="///", ec="#c8c8c8", lw=.3))
            else:
                ax.text(j, i, (texto[i][j] if texto is not None else format(V[i, j], fmt)), ha="center", va="center", fontsize=fs, color="#1a1a1a")
    ax.set_xticks(range(V.shape[1])); ax.set_xticklabels(colunas_nome, rotation=90, fontsize=6.8)
    ax.set_yticks(range(V.shape[0]))
    if sep_apos is not None: ax.axvline(sep_apos + .5, color="#1a1a1a", lw=1.2)
    limpa(ax)

fig, ax = plt.subplots(figsize=(9.6, 4.6))
heat_linhas(ax, R1A, ".1f", VERDE, True, [NOME[a] for a in ONLINE], sep_apos=len(SA_ON) - 1)
ax.set_yticklabels([f"{l}  (n={NP1A[l]})" for l in R1A.index], fontsize=7)
ax.set_title("Rank médio de IGD+ por grupo de característica — 17 configurações online (verde = melhor na linha)\n"
             "rank por problema entre as 17; média sobre os problemas do grupo; pisos à direita da linha preta · "
             + ("PROTÓTIPO (cache 17/08)" if "PROTÓTIPO" in FONTE_END else "corpus canônico"), fontsize=8, loc="left")
salva_fig(fig, "fig_1A_rank_grupo")

fig, ax = plt.subplots(figsize=(8.2, 4.6))
V = D1A.values.astype(float) * 100
lim = float(np.nanpercentile(np.abs(V), 92)) if np.isfinite(V).any() else 1
ax.imshow(np.ma.masked_invalid(np.clip(V, -lim, lim)), cmap=DIVERG, vmin=-lim, vmax=lim, aspect="auto")
for i in range(V.shape[0]):
    for j in range(V.shape[1]):
        if np.isnan(V[i, j]):
            ax.add_patch(plt.Rectangle((j - .5, i - .5), 1, 1, color="#ececec", hatch="///", ec="#c8c8c8", lw=.3))
        else:
            ax.text(j, i, f"{V[i, j]:+.0f}%\n{T1A.values[i, j]*100:.0f}v", ha="center", va="center", fontsize=5.6, color="#1a1a1a")
ax.set_xticks(range(len(SA_ON))); ax.set_xticklabels([NOME[a] for a in SA_ON], rotation=90, fontsize=6.8)
ax.set_yticks(range(len(D1A))); ax.set_yticklabels([f"{l}  (n={NP1A[l]})" for l in D1A.index], fontsize=7)
limpa(ax)
ax.set_title("Ganho sobre o piso casado por grupo — Δ = (IGD+_piso − IGD+_assistido)/IGD+_piso, mediana sobre os problemas do grupo\n"
             "célula: Δ% e taxa de vitórias por semente (v); verde = assistido melhor, laranja = piso melhor · "
             + ("PROTÓTIPO (cache 17/08)" if "PROTÓTIPO" in FONTE_END else "corpus canônico"), fontsize=8, loc="left")
salva_fig(fig, "fig_1A_delta_grupo")

print("1A · rank médio por grupo (menor = melhor):")
print(R1A.rename(columns=NOME).round(1).to_string())
print("\n1A · Δ mediano por grupo (%; >0 = assistido melhor):")
print((D1A.rename(columns=NOME) * 100).round(0).to_string())
melhor = pd.DataFrame({"melhor_rank": R1A.idxmin(axis=1).map(NOME), "melhor_delta": D1A.idxmax(axis=1).map(NOME),
                       "melhor_piso": R1A[PISOS].idxmin(axis=1).map(NOME)})
print("\n1A · melhor por grupo:"); print(melhor.to_string())

1A · rank médio por grupo (menor = melhor):
                                               CSEA  PC-SAEA  CLMEA  MMRAEA   EIM  K-RVEA  θ-DEA-DP  ParEGO  EDN-ARMOEA  qNEHVI  JES  qPOTS  LBN-MOBO  NSGA-II  MOEA/D  NSGA-III  SMS-EMOA
(1) multimodalidade                            10.9     10.2    8.2     5.1   9.0     6.8       4.4     9.2        10.3     4.1  7.3   11.1      16.3      8.4    11.6       8.8       7.8
(2) não separabilidade                         12.0     11.6    6.5     2.8   9.1     5.2       3.9     9.2         8.8     4.1  6.0   11.6      16.8      8.9    15.2      10.5       8.0
(3) enganosidade / falsos ótimos               11.5     10.3    6.7     3.7   6.7     5.2       4.2     7.5        10.5     3.8  7.7   10.2      15.0      9.2    15.2      10.3       8.8
(4) regiões planas                              3.0      6.0    9.0     7.0  11.0     5.0      15.0     8.0        10.0     NaN  NaN   14.0      13.0      2.0    12.0       4.0       1.0
(5) heterocedasticida

## 1B · ficha 18 — as escadas: em que degrau cada assistido (e cada classe) deixa de vencer o piso

Uma escada por (grupo, trilha) com ≥ 2 degraus (11 escadas) + as duas transversais (X: nº de grupos acumulados; D: dimensionalidade, com a escada controlada). Em cada degrau, o Δ mediano do assistido contra o seu piso casado (se dois problemas dividem o degrau, mediana dos dois). **[proposta]** "deixa de funcionar" = primeiro degrau com Δ mediano < 0 **e** taxa de vitórias < 50 %. Versão por classe = mediana dos algoritmos da classe de função da incerteza. O grupo (4) é lido em HV (o DDMOP7 só tem HV) — declarado no painel.

In [6]:
# ── 1B. Escadas ───────────────────────────────────────────────────────────────────────
ESC = CAR[CAR.grupo.isin(list("1234567"))].copy()
ESC["chave"] = list(zip(ESC.grupo, ESC.trilha))
ESCADAS = []
for (g, tr), s in ESC.groupby(["grupo", "trilha"], sort=True):
    degraus = (s.sort_values("degrau").groupby("degrau")
               .agg(rotulo=("degrau_rotulo", "first"), probs=("prob", list)).reset_index())
    if len(degraus) >= 2:
        ESCADAS.append(dict(grupo=g, trilha=tr, nome=f"({g}) {NOME_GRUPO[g]}" + (f" — trilha {tr.split(' — ')[0]}" if tr != "única" else ""),
                            degraus=degraus, metrica=("hv" if g == "4" else "igd_plus")))
print(len(ESCADAS), "escadas:", [e["nome"] for e in ESCADAS])

def delta_por_degrau(escada, dcel_igdp, dcel_hv, algs):
    dc = dcel_hv if escada["metrica"] == "hv" else dcel_igdp
    Dm = pd.DataFrame(index=escada["degraus"].rotulo, columns=algs, dtype=float); Tv = Dm.copy()
    for _, row in escada["degraus"].iterrows():
        sub = dc[dc.problema.isin(row.probs)]
        for a in algs:
            s = sub[sub.alg == a]
            if len(s):
                Dm.loc[row.rotulo, a] = s.delta_med.median(); Tv.loc[row.rotulo, a] = s.taxa_vit.mean()
    return Dm, Tv

CRITERIO = dict(delta_max=0.0, taxa_max=0.5)   # [proposta]
def degrau_de_falha(dcol, tcol):
    vistos = [(r, d, t) for r, d, t in zip(dcol.index, dcol.values, tcol.values) if not np.isnan(d)]
    if not vistos: return "sem dado"
    for k, (r, d, t) in enumerate(vistos):
        if d < CRITERIO["delta_max"] and (np.isnan(t) or t < CRITERIO["taxa_max"]):
            return "desde o 1º" if k == 0 else r
    return "nunca falha"

linhas_falha, linhas_classe = [], []
DEG = {}
for e in ESCADAS:
    Dm, Tv = delta_por_degrau(e, DCEL, DCEL_HV, SA_ON); DEG[e["nome"]] = (Dm, Tv)
    for a in SA_ON:
        linhas_falha.append(dict(escada=e["nome"], metrica=e["metrica"], algoritmo=NOME[a], classe_funcao=FUNCAO[a],
                                 degrau_de_falha=degrau_de_falha(Dm[a], Tv[a]),
                                 **{f"delta_{r}": Dm.loc[r, a] for r in Dm.index}))
    for cl in ["OV", "AV", "OM", "AM", "OL", "AR"]:
        algs = [a for a in SA_ON if FUNCAO[a] == cl]
        dcl, tcl = Dm[algs].median(axis=1), Tv[algs].mean(axis=1)
        linhas_classe.append(dict(escada=e["nome"], metrica=e["metrica"], classe=FUNCAO_NOME[cl], n_alg=len(algs),
                                  degrau_de_falha=degrau_de_falha(dcl, tcl), **{f"delta_{r}": dcl[r] for r in Dm.index}))
FALHA = pd.DataFrame(linhas_falha); FALHA_CL = pd.DataFrame(linhas_classe)
salva_tab(FALHA, "tab_1B_degrau_falha", index=False); salva_tab(FALHA_CL, "tab_1B_degrau_falha_classe", index=False)

# figura: painéis pequenos, uma escada por painel, linha fina por algoritmo (cor = classe), grossa = mediana da classe
ncol = 4; nrow = int(np.ceil(len(ESCADAS) / ncol))
fig, axs = plt.subplots(nrow, ncol, figsize=(13.5, 3.1 * nrow))
for ax, e in zip(axs.ravel(), ESCADAS):
    Dm, Tv = DEG[e["nome"]]; x = np.arange(len(Dm))
    linhas_escada(ax, Dm, x, SA_ON)
    ax.set_xticks(x)
    ax.set_xticklabels([f"{r}\n" + "\n".join(ROTULO_PROB[p] for p in probs) for r, probs in zip(Dm.index, e["degraus"].probs)], fontsize=5.6)
    ax.set_title(e["nome"] + ("  [HV]" if e["metrica"] == "hv" else ""), fontsize=7.2, loc="left")
    ax.tick_params(labelsize=6); ax.grid(alpha=.25, lw=.4); limpa(ax)
for ax in axs.ravel()[len(ESCADAS):]:
    ax.axis("off")
h, l = axs.ravel()[0].get_legend_handles_labels()
fig.legend(h, l, ncol=6, loc="lower center", fontsize=7, frameon=False)
fig.suptitle("Escadas de dificuldade — Δ% sobre o piso casado por degrau (linha fina = algoritmo; grossa = mediana da classe de função da incerteza; acima de 0 = assistido melhor; ▼ = abaixo de −100 %, cortado) · "
             + ("PROTÓTIPO (cache 17/08)" if "PROTÓTIPO" in FONTE_END else "corpus canônico"), fontsize=9, x=.01, ha="left")
fig.supylabel("Δ IGD+ (%)  [grupo 4: Δ HV]", fontsize=8)
fig.tight_layout(rect=[0, .05, 1, .96])
salva_fig(fig, "fig_1B_escadas")

print("1B · degrau de falha por classe (critério [proposta]: Δ mediano < 0 e vitórias < 50 %):")
print(FALHA_CL.pivot(index="escada", columns="classe", values="degrau_de_falha").to_string())

11 escadas: ['(1) multimodalidade — trilha A', '(1) multimodalidade — trilha B', '(1) multimodalidade — trilha C', '(2) não separabilidade', '(3) enganosidade / falsos ótimos — trilha enganosidade', '(3) enganosidade / falsos ótimos — trilha falso ótimo', '(4) regiões planas', '(5) heterocedasticidade / densidade irregular', '(6) geometria do front (côncavo, desconexo) — trilha côncava', '(6) geometria do front (côncavo, desconexo) — trilha desconexa', '(7) BBOB']


1B · degrau de falha por classe (critério [proposta]: Δ mediano < 0 e vitórias < 50 %):
classe                                                         AM · gestão de confiança AR · dominância probabilística AV · penalização de risco OL · ganho de informação OM · aprendizado ativo OV · aquisição exploratória
escada                                                                                                                                                                                                                      
(1) multimodalidade — trilha A                                                       A2                     desde o 1º               nunca falha                       A3             desde o 1º                 nunca falha
(1) multimodalidade — trilha B                                               desde o 1º                    nunca falha                        B2               desde o 1º             desde o 1º                  desde o 1º
(1) multimodalidade — trilha

In [7]:
# ── 1B-T. As duas escadas transversais: X (nº de grupos acumulados) e D (dimensionalidade) ─────
X = CAR[CAR.grupo == "X"][["prob", "degrau"]].rename(columns={"degrau": "n_grupos"}).drop_duplicates("prob")
Dd = CAR[CAR.grupo == "D"][["prob", "D"]].drop_duplicates("prob")
def escada_transversal(chave_df, col, dcel, algs, filtro=None):
    df = dcel.merge(chave_df, left_on="problema", right_on="prob")
    if filtro is not None: df = df[df.problema.isin(filtro)]
    Dm = df.groupby([col, "alg"]).delta_med.median().unstack().reindex(columns=algs)
    Tv = df.groupby([col, "alg"]).taxa_vit.mean().unstack().reindex(columns=algs)
    n = df.groupby(col).problema.nunique()
    return Dm, Tv, n
DX, TX, NX = escada_transversal(X, "n_grupos", DCEL, SA_ON)
DD, TD, ND = escada_transversal(Dd, "D", DCEL, SA_ON)
CONTROLADA = ["MMF1", "RE21", "BBOB_F1", "ZDT1"]       # geometria convexa fixa; só D muda
DC_, TC_, NC_ = escada_transversal(Dd, "D", DCEL, SA_ON, filtro=CONTROLADA)
RX = RANK.join(X.set_index("prob"), how="inner").groupby("n_grupos").mean()
RD = RANK.join(Dd.set_index("prob"), how="inner").groupby("D").mean()
salva_tab(DX.rename(columns=NOME).assign(n_problemas=NX), "tab_1B_transversal_cross_delta")
salva_tab(DD.rename(columns=NOME).assign(n_problemas=ND), "tab_1B_transversal_D_delta")
salva_tab(RX.rename(columns=NOME), "tab_1B_transversal_cross_rank"); salva_tab(RD.rename(columns=NOME), "tab_1B_transversal_D_rank")

fig, axs = plt.subplots(1, 3, figsize=(13.5, 3.8))
PAINEIS = [(DX, NX, "X · acúmulo de características (nº de grupos; inclui os reais)", "nº de grupos em que o problema entra", lambda v, n: f"{int(v)}\n(n={n})"),
           (DD, ND, "D · maldição da dimensionalidade (todos os 28; orçamento 31D−1)", "D", lambda v, n: f"D={int(v)}\n(n={n})"),
           (DC_, NC_, "D · escada controlada — geometria convexa fixa, só D muda", "D", lambda v, n: f"D={int(v)}\n" + ROTULO_PROB[[p for p in CONTROLADA if Dd.set_index('prob').D[p] == v][0]])]
for ax, (Dm, n, titulo, xl, rot) in zip(axs, PAINEIS):
    x = np.arange(len(Dm))
    linhas_escada(ax, Dm, x, SA_ON)
    ax.set_xticks(x); ax.set_xticklabels([rot(v, n.get(v, 0)) for v in Dm.index], fontsize=6)
    ax.set_title(titulo, fontsize=7.4, loc="left"); ax.set_xlabel(xl, fontsize=7); ax.tick_params(labelsize=6); ax.grid(alpha=.25, lw=.4); limpa(ax)
axs[0].set_ylabel("Δ IGD+ (%) sobre o piso casado", fontsize=7)
h, l = axs[0].get_legend_handles_labels(); fig.legend(h, l, ncol=6, loc="lower center", fontsize=7, frameon=False)
fig.suptitle("Escadas transversais — Δ mediano por degrau (▼ = abaixo de −100 %, cortado) · " + ("PROTÓTIPO (cache 17/08)" if "PROTÓTIPO" in FONTE_END else "corpus canônico"), fontsize=9, x=.01, ha="left")
fig.tight_layout(rect=[0, .1, 1, .95]); salva_fig(fig, "fig_1B_transversais")
print("Δ mediano (%) por nº de grupos acumulados:"); print((DX.rename(columns=NOME) * 100).round(0).assign(n=NX).to_string())
print("\nΔ mediano (%) por D:"); print((DD.rename(columns=NOME) * 100).round(0).assign(n=ND).to_string())
print("\nrank médio por D:"); print(RD.rename(columns=NOME).round(1).to_string())

Δ mediano (%) por nº de grupos acumulados:
alg       CSEA  PC-SAEA  CLMEA  MMRAEA   EIM  K-RVEA  θ-DEA-DP  ParEGO  EDN-ARMOEA  qNEHVI   JES  qPOTS  LBN-MOBO   n
n_grupos                                                                                                             
0          NaN      9.0   89.0    86.0  77.0     NaN       NaN     NaN         NaN     NaN   NaN    NaN       NaN   1
1         -3.0    -22.0   29.0    56.0  57.0    49.0      64.0    67.0         6.0    59.0 -99.0  -84.0     -80.0  10
2        -12.0    -19.0   24.0    13.0  26.0    45.0      54.0    36.0        -3.0    48.0  11.0    6.0    -137.0  11
3        -17.0    -18.0   41.0    63.0  21.0    63.0      73.0    57.0        34.0    35.0  25.0  -40.0    -299.0   3
4         -8.0     -5.0   22.0    59.0 -13.0    54.0      64.0    44.0        42.0    31.0  35.0   -5.0     -98.0   2

Δ mediano (%) por D:
alg  CSEA  PC-SAEA  CLMEA  MMRAEA   EIM  K-RVEA  θ-DEA-DP  ParEGO  EDN-ARMOEA  qNEHVI    JES  qPOTS  LBN-MOBO

## 1C-19 (metade "resultado") · as seis perguntas do cap. 4 lidas em 1A/1B

Uma linha por pergunta: o grupo/trilha que a responde, os problemas, quem vence (rank médio no grupo), o melhor piso, o Δ mediano do melhor assistido, quantos assistidos batem o piso no grupo, e o degrau de falha mais frequente por classe. A metade "mecanismo" (sonda ③) fica para a passagem única (SPEC §4.6; decisão 2 da fila).

In [8]:
# ── 1C-19. Tabela pergunta × resultado ───────────────────────────────────────────────
PERGUNTAS = [(1, "multimodalidade: a incerteza impediu a busca de ficar presa numa bacia errada?", "1", None),
             (2, "não separabilidade: quando as variáveis interagem, o modelo erra mais — e o σ sabe?", "2", None),
             (3, "paisagens enganosas: o σ avisa ou o modelo fica confiante no erro?", "3", None),
             (4, "regiões planas: no platô, a incerteza ainda dá direção à busca?", "4", None),
             (5, "frentes desconexas: o modelo alucinou pontes, e o σ marcou a ponte?", "6", "desconexa"),
             (6, "densidade não uniforme: o σ é maior onde há menos dado, e isso levou a busca à região rara?", "5", None)]
linhas = []
for k, texto, g, trilha in PERGUNTAS:
    s = CAR[(CAR.grupo == g) & ((CAR.trilha == trilha) if trilha else True)]
    probs = sorted(set(s.prob), key=PROBLEMAS.index)
    pr = [p for p in probs if p in RANK.index]
    rk = RANK.loc[pr].mean(); dm = DCEL[DCEL.problema.isin(pr)].groupby("alg").delta_med.median()
    esc = [e["nome"] for e in ESCADAS if e["grupo"] == g and (trilha is None or trilha in e["trilha"])]
    falhas = FALHA_CL[FALHA_CL.escada.isin(esc)].groupby("classe").degrau_de_falha.agg(lambda x: x.mode().iloc[0] if len(x) else "")
    linhas.append(dict(pergunta=k, texto=texto, grupo=g + (f"/{trilha}" if trilha else ""),
                       problemas=", ".join(ROTULO_PROB[p] for p in probs),
                       vence_rank=(NOME[rk.idxmin()] if rk.notna().any() else "—"),
                       melhor_piso=(NOME[rk[PISOS].idxmin()] if rk[PISOS].notna().any() else "—"),
                       melhor_delta=(f"{NOME[dm.idxmax()]} ({dm.max()*100:+.0f}%)" if len(dm) else "—"),
                       assistidos_batem_piso=f"{int((dm > 0).sum())}/{int(dm.notna().sum())}",
                       degrau_de_falha_por_classe="; ".join(f"{c.split(' · ')[0]}: {v}" for c, v in falhas.items())))
SEIS = pd.DataFrame(linhas); salva_tab(SEIS, "tab_1C_seis_perguntas", index=False)
print(SEIS.drop(columns="texto").to_string(index=False))

 pergunta       grupo                                                                   problemas vence_rank melhor_piso    melhor_delta assistidos_batem_piso                                                                       degrau_de_falha_por_classe
        1           1 ZDT4, DTLZ1, DTLZ3, WFG4, MMF1, MMF4, MMF11_L, MMF16_20, F17, F37, F49, F55     qNEHVI    SMS-EMOA θ-DEA-DP (+50%)                  7/13 AM: desde o 1º; AR: nunca falha; AV: nunca falha; OL: desde o 1º; OM: desde o 1º; OV: desde o 1º
        2           2                                     WFG2, WFG9, F5, F17, F22, F37, F49, F55     MMRAEA    SMS-EMOA θ-DEA-DP (+65%)                  8/13        AM: desde o 1º; AR: nunca falha; AV: nunca falha; OL: nunca falha; OM: nunca falha; OV: 3
        3           3                                     WFG5, WFG9, MMF11_L, MMF16_20, F49, F55     MMRAEA    SMS-EMOA θ-DEA-DP (+58%)                  8/13   AM: desde o 1º; AR: nunca falha; AV: nunca falha; OL: desde o 1º; OM: n

## Resumo do lote (números do protótipo; regenerar após a consolidação)

In [9]:
resumo = dict(fonte=FONTE_END, celulas_validas=int(len(END)),
              rank_medio_online=RANK.mean().rename(index=NOME).round(3).to_dict(),
              placar_igdp={NOME[a]: [int((s.delta_med > 0).sum()), int((s.delta_med < 0).sum())] for a, s in DCEL.groupby("alg")},
              rank_por_grupo=R1A.rename(columns=NOME).round(2).T.to_dict(),
              delta_por_grupo=(D1A.rename(columns=NOME) * 100).round(1).T.to_dict(),
              degrau_falha_classe=FALHA_CL.pivot(index="escada", columns="classe", values="degrau_de_falha").to_dict())
with open(os.path.join(OUT, "resumo_lote1.json"), "w") as f:
    json.dump(resumo, f, ensure_ascii=False, indent=1, default=str)
print("resumo salvo em", os.path.join(OUT, "resumo_lote1.json"))
print("figuras:", sorted(os.listdir(FIGDIR))); print("tabelas:", len(os.listdir(TABDIR)))

resumo salvo em /Users/gmello/Documents/python_repos/mestrado/ua-dd-saea/data/analysis_cache/lote1/resumo_lote1.json
figuras: ['fig_1A_delta_grupo.pdf', 'fig_1A_delta_grupo.png', 'fig_1A_rank_grupo.pdf', 'fig_1A_rank_grupo.png', 'fig_1B_escadas.pdf', 'fig_1B_escadas.png', 'fig_1B_transversais.pdf', 'fig_1B_transversais.png']
tabelas: 23
